# Pipeline 8 - Full Manga Translation Pipeline Test

Testing the refactored pipeline with:
- `BubbleSegmentationWithOrdering` (panel-aware bubble ordering + visualization)
- `MangaOCRModel`
- `ElanMtJaEnTranslator`
- `MangaTypesetter`
- `MangaPipeline`


In [ ]:
import os
import sys

ENDWITHS = 'Pipelines'
NOTEBOOK_DIR = os.getcwd()

if not NOTEBOOK_DIR.endswith(ENDWITHS):
    raise ValueError(f"Not in correct dir, expect end with {ENDWITHS}, but got {NOTEBOOK_DIR} instead")

BASE_DIR = os.path.join(NOTEBOOK_DIR, '..', '..', '..')
sys.path.insert(0, BASE_DIR)
print(f"Base dir: {BASE_DIR}")


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Pipeline components
from src.pipeline.SegmentationModels.YoloSeg import YoloBubbleSeg, YoloPanelSeg
from src.pipeline.SegmentationModels.BubbleSegmenterWithSplit import BubbleSegmenterWithSplit
from src.pipeline.SegmentationModels.BubbleSegmentationWithOrdering import BubbleSegmentationWithOrdering
from src.pipeline.OCRModels.MangaOCRModel import MangaOCRModel
from src.pipeline.TranslationModels.ElanMtJaEnTranslator import ElanMtJaEnTranslator
from src.pipeline.Utils.MangaTypesetter import MangaTypesetter
from src.pipeline.Utils.MangaPipeline import MangaPipeline

print("Imports successful!")


In [ ]:
# Configuration
import torch
DEVICE = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

# Example image path
EX_IMG_PATH = os.path.join(BASE_DIR, "image.png")

print(f"Device: {DEVICE}")
print(f"Image path: {EX_IMG_PATH}")
print(f"Image exists: {os.path.exists(EX_IMG_PATH)}")


## 1. Load Image


In [ ]:
# Load image
image = cv2.imread(EX_IMG_PATH)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

print(f"Image shape: {image.shape}")

# Display original image
plt.figure(figsize=(12, 16))
plt.imshow(image)
plt.title("Original Image")
plt.axis('off')
plt.show()


## 2. Create and Test Segmentation with Ordering


In [ ]:
# Create detectors
print("Creating bubble detector...")
bubble_detector = YoloBubbleSeg(variant="v8n", device=DEVICE, verbose=True)

print("\nCreating panel detector...")
panel_detector = YoloPanelSeg(variant="v8n", device=DEVICE, verbose=True)

bubble_detector_with_split = BubbleSegmenterWithSplit(
    variant="v11s",
    device=DEVICE,
    verbose=True,
    plot=True
)

# Wrap with ordering
print("\nCreating ordered segmenter...")
ordered_segmenter = BubbleSegmentationWithOrdering(
    bubble_detector=bubble_detector_with_split,
    panel_detector=panel_detector,
    num_panel_rows=4,
    right_to_left=True,
    verbose=True,
    plot=True
)

print("Segmenters created!")


In [ ]:
# Load models and run segmentation
print("Loading segmentation models...")
ordered_segmenter.load_model()

print("\nRunning segmentation...")
bboxes, masks, confs = ordered_segmenter.predict(image, conf_threshold=0.5)

print(f"Found {len(bboxes)} bubbles")
print(f"Confidences: {[f'{c:.2f}' for c in confs]}")


## 3. Visualize Segmentation Results


## 4. Test Full Pipeline


In [ ]:
# Unload segmenter first (will be loaded by pipeline)
ordered_segmenter.unload_model()

# Create full pipeline
print("Creating pipeline components...")
ocr_model = MangaOCRModel(verbose=True)
translator = ElanMtJaEnTranslator(elan_model="tiny", device=DEVICE, verbose=True)
typesetter = MangaTypesetter()

pipeline = MangaPipeline(
    segmenter=ordered_segmenter,
    ocr_model=ocr_model,
    translator=translator,
    typesetter=typesetter,
    verbose=True
)

print("Pipeline created!")


In [ ]:
# Run full pipeline with context manager
print("Running full pipeline...\n")

with pipeline:
    final_image, results = pipeline.process(image, conf_threshold=0.5, return_intermediate=True)

print("Pipeline completed!")


In [ ]:
# Display OCR and Translation results

print("PIPELINE RESULTS")

if 'ocr_texts' in results:
    print(f"\n📝 OCR Results ({len(results['ocr_texts'])} bubbles):")
    for i, text in enumerate(results['ocr_texts']):
        print(f"  [{i}] {text}")

print(f"\n🌐 Translated Texts:")
for i, text in enumerate(results['translated_texts']):
    print(f"  [{i}] {text}")


## 5. Compare Original vs Translated


In [ ]:
# Compare original vs final
fig, axes = plt.subplots(1, 2, figsize=(20, 14))

axes[0].imshow(image)
axes[0].set_title("Original Image", fontsize=14)
axes[0].axis('off')

axes[1].imshow(final_image)
axes[1].set_title("Translated Image", fontsize=14)
axes[1].axis('off')

plt.suptitle("Before vs After Translation", fontsize=16)
plt.tight_layout()
plt.show()


In [ ]:
# Display final result larger
plt.figure(figsize=(14, 18))
plt.imshow(final_image)
plt.title("Final Translated Manga Page", fontsize=16)
plt.axis('off')
plt.tight_layout()
plt.show()


## Testing Complete!

The pipeline successfully:
1. Detects speech bubbles with YOLO segmentation
2. Orders bubbles in manga reading order (right-to-left, top-to-bottom)
3. Uses panel detection for better bubble grouping
4. Performs OCR on each bubble
5. Translates Japanese to English
6. Typesets translated text back onto the image
